# Coherence Under Change: model runs on Kaggle

Same runs as the Colab notebook, for when Colab's free GPU quota is used up. Kaggle gives about 30 GPU hours a week.

**One-time setup**
1. kaggle.com → your profile → Settings → verify your phone number (needed for GPU and internet).
2. Make the project a Kaggle dataset: zip the `coherence-under-change` folder (without `zoo/`), then
   kaggle.com → Create → New Dataset → upload the zip → name it `coherence-under-change` → Create.
3. Optional, to continue earlier runs: zip your `cuc_runs` folder (from Google Drive) and upload it as a second
   dataset named `cuc-runs`. Finished runs in it are skipped.
4. In this notebook: right panel → Session options → Accelerator **GPU T4 x2**, Internet **on**;
   Add Input → your `coherence-under-change` dataset (and `cuc-runs` if you made it).

**Running without keeping the tab open:** click **Save Version → Save & Run All (Commit)**. Kaggle runs every cell
in the background (up to 12 hours); afterwards open the version → Output → download `cuc_runs.zip`.

In [ ]:
# 1. GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Project + earlier results (read-only inputs are copied to the writable /kaggle/working)
import glob, os, shutil
src = [p for p in glob.glob('/kaggle/input/**/cuc', recursive=True) if os.path.isdir(p) and os.path.exists(os.path.join(os.path.dirname(p), 'scripts'))]
assert src, 'Add your coherence-under-change dataset as an input (right panel -> Add Input)'
PROJECT = os.path.dirname(src[0])
shutil.rmtree('/kaggle/working/cuc', ignore_errors=True)
shutil.copytree(PROJECT, '/kaggle/working/cuc', ignore=shutil.ignore_patterns('runs', 'results', 'zoo', '__pycache__', '*.png'))
OUT = '/kaggle/working/cuc_runs'
prev = [p for p in glob.glob('/kaggle/input/**/cuc_runs', recursive=True) if os.path.isdir(os.path.join(p, 'runs'))]
if prev and not os.path.exists(OUT):
    shutil.copytree(prev[0], OUT)
    print('continuing from earlier runs in', prev[0])
os.makedirs(OUT, exist_ok=True)
MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'   # change both lines to run another model
NAME = 'qwen2.5-coder-7b'
%cd /kaggle/working/cuc
print('project:', PROJECT)

In [ ]:
# 3. Libraries (about 2 minutes; dependency warnings about other preinstalled packages are harmless)
!pip -q install -U transformers accelerate bitsandbytes peft trl datasets networkx

In [ ]:
# 4. Optional, only for gated models such as Llama 3.1: add your Hugging Face token as a Kaggle secret named HF_TOKEN
#    (top menu Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient; from huggingface_hub import login
# login(UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
# 5. Pipeline check without a model (seconds)
!python3 -m unittest discover -s tests 2>&1 | tail -3
!python3 scripts/run_models.py --backend gold --bed abilene --seeds 100-100 --length 10 --conds C0,C3 --out /kaggle/working/check

In [ ]:
# 6. Pilot: all four conditions (finished runs are skipped).  Two T4s: the model is split across both, so long
#    C3 prompts are less likely to run out of memory.
!python3 scripts/run_models.py --backend hf --model {MODEL} --name {NAME} --bed abilene --seeds 100-102 --length 15 --conds C0,C3,C1,C2 --workers 3 --out {OUT}

In [ ]:
# 7. Results table + paired bootstrap, and one zip to download (Output tab, or the file browser on the right)
!python3 scripts/summarize_runs.py {OUT} --out {OUT}/summary
!cd /kaggle/working && zip -qr cuc_runs.zip cuc_runs && ls -lh cuc_runs.zip